In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from collections import defaultdict

In [15]:
os.getcwd()
samples_folder = Path("./screens")
etalons_folder = Path("./symbols")
extraction_folder = Path("./extraction")
extraction_folder.mkdir(exist_ok=True)

In [16]:
symbol_width = 140
symbol_height = 140

etalons = {}
for template in etalons_folder.glob("*.png"):
    name = template.stem
    image = cv2.imread(str(template), cv2.IMREAD_COLOR_RGB)
    resized = image # cv2.resize(image, (symbol_width, symbol_height))
    etalons[name] = resized

In [17]:
reel_area = (764, 470) 
cols, rows = 5, 3

vertical_spacing = 82
horizontal_spacing = 81

def extract_symbols(image_path, output_root):
    image = Image.open(image_path)
    image_name = Path(image_path).stem 
    
    target_dir = Path(output_root) / image_name
    target_dir.mkdir(parents=True, exist_ok=True)
    
    for r in range(rows):
        for c in range(cols):
            left = reel_area[0] + (c * (symbol_width + vertical_spacing))
            top = reel_area[1] + (r * (symbol_height + horizontal_spacing))

            symbol = image.crop((left, top, left + symbol_width, top + symbol_height))
            symbol.save(target_dir / f"symbol_r{r}_c{c}.png")

In [18]:
for file_path in samples_folder.glob("*.png"):
    extract_symbols(file_path, extraction_folder)

In [19]:
def classify_images(folder_path):
    crop_folder = Path(folder_path)
    
    for crop_file in crop_folder.glob("*.png"):
        crop_image = cv2.imread(str(crop_file), cv2.IMREAD_COLOR_RGB)
        best_score = -1
        best_match = "Unknown"

        for name, template_image in etalons.items():
            result = cv2.matchTemplate(crop_image, template_image, cv2.TM_CCOEFF_NORMED)
            _, max_value, _, _ = cv2.minMaxLoc(result)

            if max_value > best_score:
                best_score = max_value
                best_match = name

        tag = best_match
        
        text_file = crop_file.with_suffix('.txt')
        
        with open(text_file, 'w') as file:
            file.write(tag)

In [20]:
for subfolder in extraction_folder.iterdir():
    if subfolder.is_dir():
        classify_images(subfolder)

In [ ]:
symbol_histogram = defaultdict(lambda: defaultdict(int))
chunk_histogram = defaultdict(lambda: defaultdict(int))

def update_symbol_histogram(file_path):
    cell_name = os.path.splitext(os.path.basename(file_path))[0]
    
    with open(file_path, 'r') as f:
        word = f.read().strip()
        if word:
            symbol_histogram[cell_name][word] += 1
            
    return word

for subdir in os.listdir(extraction_folder):
    subdir_path = os.path.join(extraction_folder, subdir)

    r, c = 0, 0
    grid = np.empty((rows, cols), dtype=object)
    if os.path.isdir(subdir_path):
        for filename in os.listdir(subdir_path):
            if filename.endswith('.txt'):
                symbol = update_symbol_histogram(os.path.join(subdir_path, filename))
                grid[r,c] = symbol
                r += 1
                if r == rows:
                    r = 0
                    c += 1
        
        for c in range(cols):
            chunk = "".join(map(str, grid[:,c]))
            chunk_histogram[f"Column {c+1}"][chunk] += 1

In [ ]:
df = pd.DataFrame.from_dict(symbol_histogram, orient='index').fillna(0).astype(int)
df = df.sort_index(axis=0).sort_index(axis=1)
#print(df)
df.to_csv(extraction_folder / 'symbol_frequencies.csv')

In [ ]:
df = pd.DataFrame.from_dict(chunk_histogram, orient='index').fillna(0).astype(int)
df = df.sort_index(axis=0).sort_index(axis=1)
df = df.T
#print(df)
df.to_csv(extraction_folder / 'chunk_frequencies.csv')